# Google Search Suggestions Analysis Tool

This notebook provides a comprehensive tool for analyzing Google search suggestions with the following capabilities:

## Features

1. **Google Suggest API Integration**: Fast retrieval of search suggestions using Google's autocomplete API
2. **Multi-Strategy Query Generation**: 
   - Base query + a-z extensions (26 variants)
   - Question words (what, how, when, where, why, who)
3. **Concurrent Processing**: Parallel execution for faster data collection
4. **Data Cleaning**: Automatic deduplication and whitespace filtering
5. **Clustering Support**: Sort and save results by cluster labels
6. **CSV Export**: Easy data export for further analysis

## Usage

```python
# Get search suggestions
results = get_suggestions_combined("heart transplant", max_workers=8, top_n=10)

# Sort by cluster
sorted_results = sort_by_cluster(cluster_results)

# Save to CSV
filename = save_to_csv(sorted_results, "output.csv")
```

## Dependencies

- requests: For API calls
- pandas: For data manipulation
- concurrent.futures: For parallel processing

In [ ]:
# Google Suggest API - Fast search suggestions retrieval
import requests
import string
import concurrent.futures
import pandas as pd
from datetime import datetime

def get_google_suggestions_fast(query, top_n=15):
    """
    Fast retrieval of Google search suggestions using Suggest API
    
    Args:
        query: Search query string
        top_n: Maximum number of suggestions to return
    
    Returns:
        list: List of search suggestions
    """
    url = "http://suggestqueries.google.com/complete/search"
    params = {
        'client': 'chrome',
        'q': query,
        'hl': 'en'
    }
    
    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            # Response format: [query, [suggestions], ...]
            data = response.json()
            suggestions = data[1] if len(data) > 1 else []
            # Return only top_n results
            return suggestions[:top_n]
    except:
        pass
    
    return []

def get_suggestions_combined(base_query, *, max_workers=10, top_n=10):
    """
    Combined strategy for fetching search suggestions with deduplication and sorting
    
    Args:
        base_query: Base search query
        max_workers: Number of concurrent workers
        top_n: Maximum suggestions per query variant
    
    Returns:
        list: Cleaned and sorted suggestions
    """
    
    # 1) Search base query itself
    base_queries = [base_query]
    
    # 2) Search base query + a-z extensions
    letters = list(string.ascii_lowercase)
    az_queries = [f"{base_query} {ch}" for ch in letters]
   
    # 3) Search base query + question words
    question_words = ['what', 'how', 'when', 'where', 'why', 'who']
    question_queries = [f"{qw} {base_query}" for qw in question_words]
    
    
    
    # 4) Combine all queries
    all_queries = list({*base_queries, *az_queries, *question_queries})
    
    print(f"Generated {len(all_queries)} query variants, max {top_n} suggestions per query")
    
    # 5) Concurrently fetch all suggestions
    all_results = set()
    
    def fetch_suggestions(query):
        try:
            return get_google_suggestions_fast(query, top_n)
        except Exception:
            return []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_suggestions, q): q for q in all_queries}
        
        for future in concurrent.futures.as_completed(futures):
            query = futures[future]
            try:
                suggestions = future.result()
                all_results.update(suggestions)
                print(f"✓ Completed: {query} ({len(suggestions)} results)")
            except Exception as exc:
                print(f"✗ Failed: {query} - {exc}")
    
    # 6) Deduplicate, filter whitespace, and sort
    clean_results = sorted({s.strip() for s in all_results if isinstance(s, str) and s.strip()})
    
    return clean_results

# 
# Test function
if __name__ == "__main__":
    # Get top 10 suggestions per query
    results = get_suggestions_combined("heart transplant", max_workers=8, top_n=10)
    print(f"\n🎉 Retrieved {len(results)} suggestions (max 10 per query):")
    for i, suggestion in enumerate(results[:20], 1):
        print(f"{i:2d}. {suggestion}")
    if len(results) > 20:
        print(f"... and {len(results) - 20} more results")


Generated 33 query variants, max 10 suggestions per query
✓ Completed: heart transplant f (10 results)
✓ Completed: who heart transplant (10 results)
✓ Completed: heart transplant s (10 results)
✓ Completed: heart transplant h (10 results)
✓ Completed: heart transplant y (10 results)
✓ Completed: heart transplant w (10 results)
✓ Completed: heart transplant g (10 results)
✓ Completed: heart transplant n (10 results)
✓ Completed: heart transplant p (10 results)
✓ Completed: heart transplant v (10 results)
✓ Completed: heart transplant o (10 results)
✓ Completed: when heart transplant (10 results)
✓ Completed: heart transplant r (10 results)
✓ Completed: heart transplant (10 results)
✓ Completed: what heart transplant (10 results)
✓ Completed: heart transplant b (10 results)
✓ Completed: heart transplant t (10 results)
✓ Completed: heart transplant q (10 results)
✓ Completed: heart transplant k (10 results)
✓ Completed: heart transplant a (10 results)
✓ Completed: heart transplant l (10 

In [22]:
# Simplified Embedding and Clustering Analysis
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import cosine_similarity

def generate_embeddings(data, model_name='all-MiniLM-L6-v2'):
    """Generate text embeddings"""
    print("Loading embedding model...")
    model = SentenceTransformer(model_name)
    print("Generating embeddings...")
    embeddings = model.encode(data)
    return embeddings

def perform_clustering(embeddings, method='kmeans', n_clusters=8, eps=0.3, min_samples=3):
    """Perform clustering algorithm"""
    print(f"Using {method} for clustering...")
    
    if method == 'kmeans':
        clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = clusterer.fit_predict(embeddings)
        n_clusters = n_clusters
    elif method == 'dbscan':
        clusterer = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = clusterer.fit_predict(embeddings)
        n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    else:
        raise ValueError(f"Unsupported clustering method: {method}")
    
    return cluster_labels, n_clusters

def create_cluster_result(suggestions, cluster_labels):
    """Create result containing original data and cluster labels"""
    result = []
    for i, (suggestion, label) in enumerate(zip(suggestions, cluster_labels)):
        result.append({
            'index': i,
            'suggestion': suggestion,
            'cluster_label': int(label)
        })
    return result

def simple_cluster_analysis(suggestions, method='kmeans', n_clusters=8, eps=0.3, min_samples=3):
    """
    Simple clustering analysis, returns only original data and cluster labels
    
    Args:
        suggestions: List of suggestions
        method: Clustering method ('kmeans' or 'dbscan')
        n_clusters: Number of clusters (for kmeans)
        eps: DBSCAN parameter
        min_samples: DBSCAN parameter
    
    Returns:
        list: Result list containing original data and cluster labels
    """
    print(f"Starting clustering analysis for {len(suggestions)} suggestions...")
    
    # 1. Generate embeddings
    embeddings = generate_embeddings(suggestions)
    
    # 2. Perform clustering
    cluster_labels, n_clusters = perform_clustering(embeddings, method, n_clusters, eps, min_samples)
    
    # 3. Create result
    result = create_cluster_result(suggestions, cluster_labels)
    
    print(f"Clustering completed! Found {n_clusters} clusters")
    return result

# Usage example
# result = simple_cluster_analysis(suggestions, method='kmeans', n_clusters=8)




/Users/mac/Desktop/Capstone/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
# Sort by Cluster and Save Data
import pandas as pd
import json
import csv
from datetime import datetime

def sort_by_cluster(cluster_results):
    """
    Sort data by cluster labels
    
    Args:
        cluster_results: List of cluster results
    
    Returns:
        list: Results sorted by cluster label and index
    """
    # Sort by cluster_label first, then by index
    sorted_results = sorted(cluster_results, key=lambda x: (x['cluster_label'], x['index']))
    return sorted_results

def save_to_csv(cluster_results, filename=None):
    """
    Save cluster results to CSV file
    
    Args:
        cluster_results: List of cluster results
        filename: Output filename, auto-generated if None
    
    Returns:
        str: Saved filename
    """
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"cluster_results_{timestamp}.csv"
    
    # Convert to DataFrame
    df = pd.DataFrame(cluster_results)
    
    # Save to CSV
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Data saved to: {filename}")
    return filename




In [36]:
cluster_results = simple_cluster_analysis(results, method='kmeans', n_clusters=8)
sorted_cluster_results = sort_by_cluster(cluster_results)
save_to_csv(sorted_cluster_results, filename='cluster_results.csv')


Starting clustering analysis for 317 suggestions...
Loading embedding model...
Generating embeddings...
Using kmeans for clustering...
Clustering completed! Found 8 clusters
Data saved to: cluster_results.csv


'cluster_results.csv'

# Labeled Data Analytics for Keyword Relevance Filtering

## Overview
**Aim**: The returned data contains many irrelevant search keys that need to be filtered according to relevance to transplant patients' needs.

**Approach**: Build a labeled dataset based on `cluster_results.csv` to train a classification model for automatic relevance filtering.

## Labeling Schema
- **Label 0**: Irrelevant - keywords that should be reduced/filtered out
- **Label 1**: Relevant - keywords that should be kept for transplant patients

## Manual Labeling Criteria
Keywords are labeled as **relevant (1)** if they address:

### Patient-Centric Information Needs:
- **Medical Information**: Survival rates, procedures, complications, recovery
- **Practical Concerns**: Cost, insurance coverage, eligibility criteria, waiting lists
- **Quality of Life**: Post-transplant care, lifestyle changes, exercise guidelines
- **Support Systems**: Support groups, family considerations, psychological aspects
- **Geographic Access**: Hospital rankings, location-specific information
- **Timeline Concerns**: Waiting periods, evaluation processes, status updates

### Examples of Relevant Keywords (Label 1):
- "heart transplant survival rate"
- "heart transplant cost with insurance"
- "heart transplant eligibility criteria"
- "heart transplant after care"
- "heart transplant exercise guidelines"
- "heart transplant waiting list"

### Examples of Irrelevant Keywords (Label 0):
- Historical facts ("first heart transplant year")
- Geographic trivia ("heart transplant in zambia")
- Non-patient focused ("heart transplant inventor")
- Academic/research focused ("heart transplant research papers")

## Implementation Pipeline

### Step 1: Manual Labeling Process
1. Load `cluster_results.csv`
2. Review each keyword through patient lens
3. Apply labeling criteria consistently
4. Create `labeled_dataset.csv` with columns: `index`, `suggestion`, `cluster_label`, `relevance_label`

### Step 2: Model Training
1. **Feature Engineering**: Use embeddings from SentenceTransformer
2. **Classification Model**: Train on labeled data (Logistic Regression, Random Forest, or Neural Network)
3. **Validation**: Cross-validation to ensure model generalizability
4. **Threshold Tuning**: Optimize decision boundary for relevance classification

### Step 3: Automated Filtering
1. **Prediction**: Apply trained model to new transplant types (liver, kidney, etc.)
2. **Filtering**: Remove keywords predicted as irrelevant (label 0)
3. **Quality Control**: Manual review of filtered results for accuracy

## Expected Outcomes
- **Reduced Noise**: Eliminate 30-50% of irrelevant keywords
- **Patient-Focused Results**: Maintain only clinically and practically relevant information
- **Scalability**: Automated filtering for multiple transplant types
- **Consistency**: Standardized relevance assessment across different organ types

## Data Quality Metrics
- **Precision**: % of kept keywords that are truly relevant
- **Recall**: % of relevant keywords that are correctly identified
- **F1-Score**: Harmonic mean of precision and recall
- **Inter-annotator Agreement**: Consistency of manual labeling (if multiple annotators)

In [ ]:

df = pd.read_csv('labeled_cluster_results.csv')  


df = df[["suggestion", "label"]]

embeddings = generate_embeddings(df["suggestion"].tolist())
print(f"Generated embeddings shape: {embeddings.shape}")
print(f"DataFrame shape: {df.shape}")


X = embeddings  
y = df["label"].values 

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution: {np.bincount(y)}")


Loading embedding model...
Generating embeddings...
Generated embeddings shape: (317, 384)
DataFrame shape: (317, 2)
Features shape: (317, 384)
Labels shape: (317,)
Label distribution: [ 76 241]


In [ ]:

# Handle Class Imbalance Problem
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate class weights
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train
)
print(f"Class weights: {dict(zip(np.unique(y_train), class_weights))}")

# 2. Retrain models with class weights
weighted_models = {
    'Weighted Logistic Regression': LogisticRegression(
        random_state=42, 
        max_iter=1000,
        class_weight='balanced'
    ),
    'Weighted Random Forest': RandomForestClassifier(
        random_state=42, 
        n_estimators=100,
        class_weight='balanced'
    )
}

print("\n=== Retraining with Class Weights ===")
weighted_results = {}
for name, model in weighted_models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Evaluate performance
    accuracy = accuracy_score(y_test, y_pred)
    weighted_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred
    }
    
    print(f"Accuracy: {accuracy:.3f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")

# 3. Compare results
print("\n=== Model Comparison ===")
print("Original vs Weighted Models:")
for name in ['Logistic Regression', 'Random Forest']:
    original_acc = results[name]['accuracy']
    weighted_name = f'Weighted {name}'
    weighted_acc = weighted_results[weighted_name]['accuracy']
    print(f"{name}: {original_acc:.3f} -> {weighted_acc:.3f}")

# 4. Select best weighted model
best_weighted_name = max(weighted_results.keys(), key=lambda k: weighted_results[k]['accuracy'])
best_weighted_model = weighted_results[best_weighted_name]['model']
print(f"\nBest Weighted Model: {best_weighted_name} (Accuracy: {weighted_results[best_weighted_name]['accuracy']:.3f})")


Class weights: {np.int64(0): np.float64(2.0737704918032787), np.int64(1): np.float64(0.6588541666666666)}

=== Retraining with Class Weights ===

Training Weighted Logistic Regression...
Accuracy: 0.859
Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.80      0.73        15
           1       0.93      0.88      0.91        49

    accuracy                           0.86        64
   macro avg       0.80      0.84      0.82        64
weighted avg       0.87      0.86      0.86        64

Confusion Matrix:
[[12  3]
 [ 6 43]]

Training Weighted Random Forest...
Accuracy: 0.859
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.40      0.57        15
           1       0.84      1.00      0.92        49

    accuracy                           0.86        64
   macro avg       0.92      0.70      0.74        64
weighted avg       0.88      0.86      0.84        64

Confusio

In [38]:
# Save Best Weighted Model to Pickle
import pickle
import joblib
from datetime import datetime

# Save the best weighted model
model_filename = f'best_weighted_model_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'
joblib.dump(best_weighted_model, model_filename)
print(f"✓ Model saved to: {model_filename}")

# Also save the embedding model for future use
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_filename = f'embedding_model_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'
joblib.dump(embedding_model, embedding_filename)
print(f"✓ Embedding model saved to: {embedding_filename}")


✓ Model saved to: best_weighted_model_20250926_192011.pkl
✓ Embedding model saved to: embedding_model_20250926_192012.pkl


In [40]:

# Save model metadata
metadata = {
    'model_name': best_weighted_name,
    'accuracy': weighted_results[best_weighted_name]['accuracy'],
    'training_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'n_samples': len(X_train),
    'class_distribution': np.bincount(y_train).tolist(),
    'feature_dimension': X_train.shape[1]
}

metadata_filename = f'model_metadata_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
import json
with open(metadata_filename, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Model metadata saved to: {metadata_filename}")

print(f"\n📁 Files created:")
print(f"• Model: {model_filename}")
print(f"• Embedding: {embedding_filename}")
print(f"• Metadata: {metadata_filename}")


✓ Model metadata saved to: model_metadata_20250926_192047.json

📁 Files created:
• Model: best_weighted_model_20250926_192011.pkl
• Embedding: embedding_model_20250926_192012.pkl
• Metadata: model_metadata_20250926_192047.json
